In [8]:
# Задание 6: Алгоритм Гровера с несколькими решениями
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import Aer
import numpy as np

print("ЗАДАНИЕ 6: АЛГОРИТМ ГРОВЕРА С M РЕШЕНИЯМИ")

n = 6  # N = 2^6 = 64
N = 2**n
rng = np.random.RandomState(42)

def build_multi_oracle(n, marked_list):
    """
    Oracle для нескольких отмеченных состояний.
    U_w|x> = -|x> если x ∈ marked_list
    """
    qc = QuantumCircuit(n)

    for marked_str in marked_list:
        marked_bits = [int(b) for b in marked_str[::-1]]

        # Применяем X к кубитам, которые должны быть 0
        for i in range(n):
            if marked_bits[i] == 0:
                qc.x(i)

        # MCZ
        qc.h(n-1)
        controls = list(range(n-1))
        if len(controls) == 1:
            qc.cx(controls[0], n-1)
        elif len(controls) == 2:
            qc.ccx(controls[0], controls[1], n-1)
        else:
            try:
                qc.mcp(np.pi, controls, n-1)
            except:
                qc.ccx(controls[0], controls[1], n-1)
        qc.h(n-1)

        # Обратно X
        for i in range(n):
            if marked_bits[i] == 0:
                qc.x(i)

    return qc

def build_diffuser_generic(n):
    """Diffuser для n кубитов"""
    qc = QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
    for i in range(n):
        qc.x(i)
    qc.h(n-1)
    controls = list(range(n-1))
    if len(controls) == 1:
        qc.cx(controls[0], n-1)
    elif len(controls) == 2:
        qc.ccx(controls[0], controls[1], n-1)
    else:
        try:
            qc.mcp(np.pi, controls, n-1)
        except:
            qc.ccx(controls[0], controls[1], n-1)
    qc.h(n-1)
    for i in range(n):
        qc.x(i)
    for i in range(n):
        qc.h(i)
    return qc

# 6.1 Для M ∈ {1, 2, 3, 4, 8}

print("\n--- 6.1 P(k) для разных M ---")

M_values = [1, 2, 3, 4, 8]
diffuser = build_diffuser_generic(n)

results_by_M = {}

for M in M_values:
    # Генерируем M случайных отмеченных состояний без повторов
    marked_indices = rng.choice(N, size=M, replace=False)
    marked_list = [format(idx, f'0{n}b') for idx in marked_indices]

    oracle = build_multi_oracle(n, marked_list)

    P_k = []
    for k in range(16):
        qc_k = QuantumCircuit(n)
        for i in range(n):
            qc_k.h(i)
        for _ in range(k):
            qc_k.compose(oracle, inplace=True)
            qc_k.compose(diffuser, inplace=True)

        sv = Statevector(qc_k)

        # Вероятность любого из отмеченных состояний
        p_total = sum(abs(sv.data[idx])**2 for idx in marked_indices)
        P_k.append(p_total)

    results_by_M[M] = {
        'marked': marked_list,
        'indices': marked_indices,
        'P_k': P_k
    }

    print(f"\nM={M}, отмеченные: {[m for m in marked_list[:3]]}{'...' if M>3 else ''}")
    print(f"  P(k): {[round(p, 4) for p in P_k[:8]]}...")


# 6.2 Первый локальный максимум vs теория

print("\n--- 6.2 Сравнение с теорией ⌊π/4·√(N/M)⌋ ---")
print(f"\n{'M':<4} {'k_теор':<8} {'k_перв.макс':<12} {'P в максимуме':<15}")
print("-"*45)

table_62 = []
for M in M_values:
    P_k = results_by_M[M]['P_k']
    k_theory = int(np.floor((np.pi/4) * np.sqrt(N/M)))

    # Найти первый локальный максимум
    k_first_max = 0
    for i in range(1, len(P_k)-1):
        if P_k[i] > P_k[i-1] and P_k[i] >= P_k[i+1]:
            k_first_max = i
            break

    p_at_max = P_k[k_first_max]
    table_62.append((M, k_theory, k_first_max, p_at_max))
    print(f"{M:<4} {k_theory:<8} {k_first_max:<12} {p_at_max:<15.6f}")


# 6.3 Глобальный максимум vs первый локальный

print("\n--- 6.3 Глобальный максимум на отрезке 0..15 ---")

for M in M_values:
    P_k = results_by_M[M]['P_k']
    k_global_max = int(np.argmax(P_k))
    p_global = max(P_k)

    # Первый локальный максимум
    k_local_max = 0
    for i in range(1, len(P_k)-1):
        if P_k[i] > P_k[i-1] and P_k[i] >= P_k[i+1]:
            k_local_max = i
            break

    differs = "Да" if k_global_max != k_local_max else "Нет"
    print(f"M={M}: k_глоб={k_global_max} (P={p_global:.4f}), k_локал={k_local_max}, различаются: {differs}")

print("""
Объяснение: После оптимума вероятность начинает расти снова, потому что
алгоритм Гровера — это вращение в двумерной плоскости. Амплитуда проходит
максимум, затем уменьшается, достигает минимума, и снова растёт.
Это периодическое поведение.

Правило выбора оптимального числа итераций:
k_opt = ⌊(π/4)·√(N/M)⌋ или ближайшее целое, дающее максимум P(k).
Если доступен только интервал 0..15, выбираем argmax_{k∈[0,15]} P(k).
""")


# 6.4 Случаи M = N/2 и M = 3N/4

print("--- 6.4 Большие M ---")

for M_frac, label in [(N//2, "N/2"), (3*N//4, "3N/4")]:
    marked_indices = rng.choice(N, size=M_frac, replace=False)
    marked_list = [format(idx, f'0{n}b') for idx in marked_indices]

    oracle = build_multi_oracle(n, marked_list)

    P_k = []
    for k in range(16):
        qc_k = QuantumCircuit(n)
        for i in range(n):
            qc_k.h(i)
        for _ in range(k):
            qc_k.compose(oracle, inplace=True)
            qc_k.compose(diffuser, inplace=True)

        sv = Statevector(qc_k)
        p_total = sum(abs(sv.data[idx])**2 for idx in marked_indices)
        P_k.append(p_total)

    k_best = int(np.argmax(P_k))
    p_best = max(P_k)
    print(f"M={label}={M_frac}: k_опт={k_best}, P={p_best:.6f}")
    print(f"  P(k): {[round(p, 4) for p in P_k[:8]]}...")

print("""
Вывод: Когда M велико (M ≥ N/2), алгоритм Гровера практически бесполезен,
так как вероятность успеха уже высока при k=0. Квадратичное ускорение
(Grover) полезно когда M << N, т.е. решение редкое.
""")

ЗАДАНИЕ 6: АЛГОРИТМ ГРОВЕРА С M РЕШЕНИЯМИ

--- 6.1 P(k) для разных M ---

M=1, отмеченные: ['110100']
  P(k): [np.float64(0.0156), np.float64(0.0156), np.float64(0.1292), np.float64(0.1083), np.float64(0.2709), np.float64(0.191), np.float64(0.3233), np.float64(0.1784)]...

M=2, отмеченные: ['011010', '100101']
  P(k): [np.float64(0.0312), np.float64(0.0312), np.float64(0.1338), np.float64(0.1995), np.float64(0.2622), np.float64(0.2581), np.float64(0.1862), np.float64(0.1204)]...

M=3, отмеченные: ['011011', '110011', '010100']
  P(k): [np.float64(0.0469), np.float64(0.0469), np.float64(0.1389), np.float64(0.2745), np.float64(0.2333), np.float64(0.2511), np.float64(0.0646), np.float64(0.055)]...

M=4, отмеченные: ['011110', '001101', '010001']...
  P(k): [np.float64(0.0625), np.float64(0.0625), np.float64(0.0625), np.float64(0.3906), np.float64(0.165), np.float64(0.274), np.float64(0.0052), np.float64(0.0016)]...

M=8, отмеченные: ['100001', '001001', '101010']...
  P(k): [np.float64(0.